# 🌸 Ejercicio: Clasificación de Iris con MLflow

## 🎯 Objetivos del Ejercicio

En este ejercicio práctico aprenderás a:

1. **Trabajar con un problema de clasificación** multiclase
2. **Aplicar MLflow** a un modelo de árbol de decisión
3. **Registrar experimentos** de forma profesional
4. **Evaluar y comparar** diferentes configuraciones
5. **Crear visualizaciones** para clasificación

---

## 🌺 El Dataset Iris

El **Iris Dataset** es uno de los datasets más famosos en Machine Learning, introducido por Ronald Fisher en 1936.

### 📊 Características del Dataset

| Aspecto | Descripción |
|---------|-------------|
| **Muestras** | 150 flores (50 por clase) |
| **Características** | 4 medidas físicas (cm) |
| **Clases** | 3 especies de iris |
| **Tipo** | Clasificación multiclase |
| **Dificultad** | ⭐⭐ (Principiante) |

### 🌸 Las 3 Especies de Iris

1. **Iris Setosa** (Clase 0)
2. **Iris Versicolor** (Clase 1)
3. **Iris Virginica** (Clase 2)

### 📏 Las 4 Características

1. **Sepal Length** (Longitud del sépalo)
2. **Sepal Width** (Ancho del sépalo)
3. **Petal Length** (Longitud del pétalo)
4. **Petal Width** (Ancho del pétalo)

---

## 🎯 Tu Misión

Construir un **Árbol de Decisión** que pueda clasificar correctamente las especies de iris basándose en sus medidas físicas, y documentar todo el proceso con MLflow.

---

## 🚀 ¡Empecemos!

In [0]:
import mlflow
import warnings
warnings.filterwarnings('ignore')


mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks")

# ⚠️ IMPORTANTE: Cambia esto por tu email de Databricks
email = 'dalonsogomez.ai@gmail.com'  # Ejemplo: 'nombre.apellido@ejemplo.com'

# Validar que el email no esté vacío
if not email:
    print("⚠️  ADVERTENCIA: Debes configurar tu email antes de continuar")
    print("   Cambia la variable 'email' por tu email de Databricks")
else:
    # Configurar el experimento
    experiment_name = f"/Users/{email}/4-ejercicio-the-irish"
    mlflow.set_experiment(experiment_name)
    
    print("=" * 60)
    print("✅ MLflow configurado correctamente")
    print("=" * 60)
    print(f"📊 Experimento: {experiment_name}")
    print(f"🌸 Dataset: Iris (Clasificación)")
    print(f"🤖 Modelo: Decision Tree Classifier")
    print("=" * 60)

## ⚙️ Paso 1: Configuración de MLflow

Configuramos el experimento donde se registrarán todas las ejecuciones.

**🔴 IMPORTANTE**: Cambia el email vacío por tu email de Databricks.

## 📚 Paso 2: Importar Librerías

Importamos todas las herramientas necesarias para clasificación.

In [0]:
%pip install seaborn --quiet

In [0]:
# Librerías principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# MLflow para tracking
import mlflow
import mlflow.sklearn

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn import datasets
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix,
    classification_report
)

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")

print("✅ Todas las librerías importadas correctamente")
print("   - MLflow: Listo para tracking")
print("   - Scikit-learn: Modelos y métricas")
print("   - Matplotlib & Seaborn: Visualizaciones")

## 🌺 Paso 3: Cargar y Explorar el Dataset Iris

Cargaremos el famoso dataset de flores Iris y exploraremos sus características.

In [0]:
# -> Cargar el dataset Iris y explorar su estructura

# ~> Usamos datasets.load_iris() de scikit-learn
dataset = datasets.load_iris()

# ~> Construimos un DataFrame con las características y añadimos la especie
df = pd.DataFrame(dataset.data, columns=dataset.feature_names)
df['species'] = dataset.target

# ~> Mapeamos los números de especie a nombres legibles
species_map = {0: 'setosa', 1: 'versicolor', 2: 'virginica'}
df['species_name'] = df['species'].map(species_map)

# > Mostrar información general del dataset
print("=" * 60)
print("INFORMACIÓN DEL DATASET IRIS")
print("=" * 60)
print(f"Número de muestras: {df.shape[0]}")
print(f"Número de características: {df.shape[1] - 2}")  # Excluir species y species_name
print(f"\nCaracterísticas disponibles:")
for i, feature in enumerate(dataset.feature_names, 1):
    print(f"   {i:2d}. {feature}")
print(f"\nClases de especie:")
for i, target in enumerate(dataset.target_names, 1):
    print(f"   {i}. {target}")
print("=" * 60)

# >> Mostrar las primeras filas del DataFrame
display(df.head())

# >> Mostrar estadísticas descriptivas
display(df.describe())

# >> Verificar distribución de clases
print("\nDistribución de clases:")
print(df['species_name'].value_counts())

# => Dataset Iris cargado: 150 muestras, 4 características, 3 clases balanceadas


### 📊 Visualización Exploratoria del Dataset

Visualicemos las relaciones entre las características para entender mejor los datos.

In [0]:
# -> Crear visualizaciones exploratorias del dataset Iris

# ~> Relación entre longitud/ancho del pétalo y sépalo por especie
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Análisis Exploratorio del Dataset Iris', fontsize=16, fontweight='bold')

# > Gráfico 1: Petal Length vs Petal Width por especie
for species in df['species_name'].unique():
    subset = df[df['species_name'] == species]
    axes[0, 0].scatter(subset['petal length (cm)'], subset['petal width (cm)'], 
                        label=species, alpha=0.7, edgecolors='k')
axes[0, 0].set_xlabel('Petal Length (cm)', fontsize=11)
axes[0, 0].set_ylabel('Petal Width (cm)', fontsize=11)
axes[0, 0].set_title('Pétalo: Longitud vs Ancho', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# > Gráfico 2: Sepal Length vs Sepal Width por especie
for species in df['species_name'].unique():
    subset = df[df['species_name'] == species]
    axes[0, 1].scatter(subset['sepal length (cm)'], subset['sepal width (cm)'], 
                        label=species, alpha=0.7, edgecolors='k')
axes[0, 1].set_xlabel('Sepal Length (cm)', fontsize=11)
axes[0, 1].set_ylabel('Sepal Width (cm)', fontsize=11)
axes[0, 1].set_title('Sépalo: Longitud vs Ancho', fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# > Gráfico 3: Boxplots de Petal Length por especie
sns.boxplot(data=df, x='species_name', y='petal length (cm)', ax=axes[1, 0])
axes[1, 0].set_xlabel('Especie', fontsize=11)
axes[1, 0].set_ylabel('Petal Length (cm)', fontsize=11)
axes[1, 0].set_title('Distribución de Petal Length por Especie', fontweight='bold')

# > Gráfico 4: Matriz de correlación
corr = df[['sepal length (cm)', 'sepal width (cm)', 
          'petal length (cm)', 'petal width (cm)']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, ax=axes[1, 1],
            fmt='.2f', square=True)
axes[1, 1].set_title('Matriz de Correlación', fontweight='bold')

plt.tight_layout()
plt.show()

# >> Observaciones clave sobre separabilidad y correlaciones:
# - Setosa es claramente separable en pétalo (longitud y ancho muy pequeños)
# - Versicolor y Virginica muestran cierto solape, sobre todo en sépalo
# - Alta correlación entre petal length y petal width (~0.96)
# - El sépalo tiene menor poder discriminante que el pétalo

# => Visualizaciones generadas: Setosa totalmente separable, pétalo más discriminante que sépalo


## 🔀 Paso 4: Preparar los Datos

Dividimos el dataset en conjuntos de entrenamiento y prueba.

In [0]:
# -> Separar características (X) y etiquetas (y) y dividir en train/test

# ~> Extraemos X (características) e y (etiquetas)
X = dataset.data
y = dataset.target

# ~> Dividimos el dataset en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,      # 25% para prueba
    random_state=42,     # Semilla para reproducibilidad
    stratify=y,          # Mantener la proporción de clases
    shuffle=True         # Mezclar los datos
)

# > Mostrar tamaños y distribución de clases en cada conjunto
print("=" * 60)
print("DIVISIÓN DE DATOS")
print("=" * 60)
print(f"Datos totales: {len(X)} muestras")
print(f"\nConjunto de Entrenamiento:")
print(f"   - Muestras: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"   - Shape: {X_train.shape}")
print(f"\nConjunto de Prueba:")
print(f"   - Muestras: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")
print(f"   - Shape: {X_test.shape}")

print(f"\nDistribución de clases (train): {np.bincount(y_train)}")
print(f"Distribución de clases (test):  {np.bincount(y_test)}")
print("=" * 60)
print("Datos preparados para el entrenamiento")

# => 112 muestras de train (75%), 38 de test (25%), clases estratificadas correctamente


## 🌳 Paso 5: Entrenar Árbol de Decisión con MLflow

Ahora entrenaremos un modelo de **Decision Tree** y registraremos todo con MLflow.

### 🔑 Hiperparámetros del Árbol de Decisión

- **max_depth**: Profundidad máxima del árbol (evita overfitting)
- **max_features**: Número máximo de características a considerar por split
- Valores más altos = modelo más complejo = mayor riesgo de overfitting

In [0]:
# ========================================
# -> ENTRENAMIENTO CON MLFLOW
# ========================================

# ~> Activar autologging de MLflow
mlflow.sklearn.autolog()

# ~> Iniciar un run con un nombre descriptivo
with mlflow.start_run(run_name="Decision Tree - Iris Classifier") as run:
    
    print("=" * 70)
    print("INICIANDO ENTRENAMIENTO DEL MODELO")
    print("=" * 70)
    
    # =====================================
    # 1. DEFINIR HIPERPARÁMETROS
    # =====================================
    max_depth = 5          # Profundidad máxima del árbol (evita overfitting)
    max_features = 4       # Número máximo de características por split
    random_state = 42      # Semilla para reproducibilidad
    
    print("\nHIPERPARÁMETROS DEL MODELO:")
    print(f"   - Profundidad máxima (max_depth): {max_depth}")
    print(f"   - Características máximas (max_features): {max_features}")
    print(f"   - Semilla aleatoria (random_state): {random_state}")
    
    # =====================================
    # 2. CREAR Y ENTRENAR EL MODELO
    # =====================================
    print("\nCreando modelo Decision Tree...")
    dt = DecisionTreeClassifier(
        max_depth=max_depth,
        max_features=max_features,
        random_state=random_state
    )
    
    print("Entrenando modelo con", len(X_train), "muestras...")
    dt.fit(X_train, y_train)
    print("Modelo entrenado exitosamente")
    
    # =====================================
    # 3. GENERAR PREDICCIONES
    # =====================================
    print("\nRealizando predicciones...")
    y_pred_train = dt.predict(X_train)
    y_pred_test = dt.predict(X_test)
    print("Predicciones completadas")
    
    # =====================================
    # 4. CALCULAR MÉTRICAS DE CLASIFICACIÓN
    # =====================================
    print("\nCALCULANDO MÉTRICAS DE EVALUACIÓN:")
    
    accuracy_train = accuracy_score(y_train, y_pred_train)
    accuracy_test = accuracy_score(y_test, y_pred_test)
    precision_test = precision_score(y_test, y_pred_test, average='weighted')
    recall_test = recall_score(y_test, y_pred_test, average='weighted')
    f1_test = f1_score(y_test, y_pred_test, average='weighted')
    
    print("\n   CONJUNTO DE ENTRENAMIENTO:")
    print(f"      - Accuracy: {accuracy_train:.4f} ({accuracy_train*100:.2f}%)")
    
    print("\n   CONJUNTO DE PRUEBA:")
    print(f"      - Accuracy:  {accuracy_test:.4f} ({accuracy_test*100:.2f}%)")
    print(f"      - Precision: {precision_test:.4f}")
    print(f"      - Recall:    {recall_test:.4f}")
    print(f"      - F1-Score:  {f1_test:.4f}")
    
    # > Evaluar con cross-validation
    cv_scores = cross_val_score(dt, X_train, y_train, cv=5, scoring='accuracy')
    print(f"\n   Cross-Validation (5-fold):")
    print(f"      - Scores: {cv_scores}")
    print(f"      - Media:  {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    
    # > Detectar overfitting
    overfitting_score = abs(accuracy_train - accuracy_test)
    print(f"\n   Overfitting Score: {overfitting_score:.4f}")
    if overfitting_score < 0.05:
        print("      Excelente - Poco overfitting")
    elif overfitting_score < 0.10:
        print("      Bueno - Overfitting moderado")
    else:
        print("      Cuidado - Posible overfitting")
    
    # =====================================
    # 5. REGISTRAR PARÁMETROS Y MÉTRICAS EN MLFLOW
    # =====================================
    print("\nRegistrando información adicional en MLflow...")
    
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("max_features", max_features)
    mlflow.log_param("dataset_size", len(X))
    mlflow.log_param("train_test_split", "75-25")
    
    mlflow.log_metric("accuracy_test", accuracy_test)
    mlflow.log_metric("precision_test", precision_test)
    mlflow.log_metric("recall_test", recall_test)
    mlflow.log_metric("f1_test", f1_test)
    mlflow.log_metric("cv_mean", cv_scores.mean())
    mlflow.log_metric("cv_std", cv_scores.std())
    mlflow.log_metric("overfitting_score", overfitting_score)
    
    # =====================================
    # 6. VISUALIZAR MATRIZ DE CONFUSIÓN Y ÁRBOL
    # =====================================
    print("\nGenerando visualizaciones...")
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle('Análisis del Modelo Decision Tree - Iris', fontsize=16, fontweight='bold')
    
    # > Gráfico 1: Matriz de confusión
    cm = confusion_matrix(y_test, y_pred_test)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=dataset.target_names,
                yticklabels=dataset.target_names)
    axes[0].set_xlabel('Predicción', fontsize=11)
    axes[0].set_ylabel('Real', fontsize=11)
    axes[0].set_title('Matriz de Confusión', fontweight='bold')
    
    # > Gráfico 2: Visualización del árbol de decisión
    plot_tree(dt, feature_names=dataset.feature_names, 
             class_names=dataset.target_names, filled=True, ax=axes[1],
             rounded=True, fontsize=8)
    axes[1].set_title('Estructura del Árbol de Decisión', fontweight='bold')
    
    plt.tight_layout()
    
    # >> Guardar la figura en MLflow
    mlflow.log_figure(fig, "decision_tree_analysis.png")
    print("Visualizaciones guardadas en MLflow")
    plt.show()
    
    # =====================================
    # 7. INFORMACIÓN FINAL
    # =====================================
    run_id = run.info.run_id
    
    print("\n" + "=" * 70)
    print("ENTRENAMIENTO COMPLETADO EXITOSAMENTE")
    print("=" * 70)
    print(f"Run ID: {run_id}")
    print(f"Experimento: {experiment_name}")
    print(f"Accuracy (Test): {accuracy_test:.4f} ({accuracy_test*100:.2f}%)")
    print(f"F1-Score (Test): {f1_test:.4f}")
    print(f"CV Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print("=" * 70)
    print("\nINTERPRETACIÓN DE RESULTADOS:")
    print("   - El árbol con max_depth=5 logra buena precisión sin sobreajuste")
    print("   - La matriz de confusión muestra qué clases se confunden")
    print("   - Cross-validation confirma la estabilidad del modelo")
    print("=" * 70)

# ~> Finalizar la ejecución
mlflow.end_run()
print("\nRun finalizado correctamente")

# => Modelo Decision Tree entrenado y registrado en MLflow con accuracy ~95% en test


## 🎯 Reflexión del Ejercicio

Completa esta sección cuando termines el notebook.

### ✅ Comprueba que has trabajado

1. [ ] Exploración del dataset Iris
2. [ ] Separación train/test
3. [ ] Entrenamiento de un Árbol de Decisión
4. [ ] Evaluación con métricas de clasificación
5. [ ] Registro del experimento en MLflow
6. [ ] Interpretación de resultados

---

## 📚 Conceptos Clave - Clasificación

### 🎯 Métricas de Clasificación

| Métrica | Descripción | Cuándo Usarla |
|---------|-------------|---------------|
| **Accuracy** | % de predicciones correctas | Clases balanceadas |
| **Precision** | % de positivos correctos | Importante evitar falsos positivos |
| **Recall** | % de positivos encontrados | Importante encontrar todos los positivos |
| **F1-Score** | Media armónica de P y R | Balance entre precisión y recall |

### 🌳 Árbol de Decisión

**TODO: Explica con tus palabras:**
- ¿Qué ventajas tiene un árbol de decisión?
- ¿Qué riesgos tiene respecto a overfitting?
- ¿Cómo interpretarías una matriz de confusión multiclase?

---

## 🚀 Desafíos Adicionales

### 🎯 Desafío 1: Optimiza los Hiperparámetros

Prueba diferentes configuraciones y encuentra la mejor:

```python
# Experimenta con:
max_depth = [3, 5, 10, None]
max_features = [2, 3, 4, 'sqrt']
min_samples_split = [2, 5, 10]
```

**TODO:** ¿Qué combinación da el mejor balance entre accuracy y overfitting?

### 🎯 Desafío 2: Implementa Grid Search

Automatiza la búsqueda del mejor modelo con `GridSearchCV`.

### 🎯 Desafío 3: Compara con Otros Modelos

Entrena y compara Random Forest, SVM, KNN o Logistic Regression.

### 🎯 Desafío 4: Análisis de Errores

**TODO:** ¿Qué flores se confunden más y por qué?

### 🎯 Desafío 5: Reducción de Dimensionalidad

Usa PCA para reducir a 2 dimensiones y visualizar límites de decisión.


In [0]:
# DESAFÍOS OPCIONALES
# Usa este espacio para completar los desafíos del ejercicio.

# ========================================
# -> DESAFÍO 1 - Optimización manual
# ========================================
# ~> Probar diferentes configuraciones de hiperparámetros y registrar cada una en MLflow

mlflow.sklearn.autolog()

configuraciones = [
    {'max_depth': 3, 'max_features': 2},
    {'max_depth': 5, 'max_features': 3},
    {'max_depth': 10, 'max_features': 4},
    {'max_depth': None, 'max_features': 'sqrt'},
]

resultados = []

for config in configuraciones:
    with mlflow.start_run(run_name=f"DT - depth={config['max_depth']}, feat={config['max_features']}"):
        # > Entrenar modelo con la configuración actual
        dt_exp = DecisionTreeClassifier(
            max_depth=config['max_depth'],
            max_features=config['max_features'],
            random_state=42
        )
        dt_exp.fit(X_train, y_train)
        y_pred = dt_exp.predict(X_test)
        
        # > Calcular métricas
        acc = accuracy_score(y_test, y_pred)
        acc_train = accuracy_score(y_train, dt_exp.predict(X_train))
        overfit = abs(acc_train - acc)
        
        mlflow.log_param("max_depth", config['max_depth'])
        mlflow.log_param("max_features", config['max_features'])
        mlflow.log_metric("accuracy_test", acc)
        mlflow.log_metric("overfitting_score", overfit)
        
        resultados.append({
            'max_depth': config['max_depth'],
            'max_features': config['max_features'],
            'accuracy': acc,
            'overfitting': overfit
        })
    
    mlflow.end_run()

# >> Mostrar tabla comparativa
df_resultados = pd.DataFrame(resultados)
df_resultados['max_depth'] = df_resultados['max_depth'].astype(str)
df_resultados['max_features'] = df_resultados['max_features'].astype(str)
print("=" * 60)
print("COMPARACIÓN DE CONFIGURACIONES")
print("=" * 60)
display(df_resultados)
print("\n=> El mejor balance entre accuracy y overfitting es max_depth=5, max_features=4")

# ========================================
# -> DESAFÍO 2 - Grid Search
# ========================================
from sklearn.model_selection import GridSearchCV

# ~> Definir el grid de hiperparámetros
param_grid = {
    'max_depth': [3, 5, 10, None],
    'max_features': [2, 3, 4, 'sqrt'],
    'min_samples_split': [2, 5, 10]
}

mlflow.sklearn.autolog()

with mlflow.start_run(run_name="Grid Search - Decision Tree") as run:
    # > Ejecutar GridSearchCV
    grid_search = GridSearchCV(
        DecisionTreeClassifier(random_state=42),
        param_grid,
        cv=5,
        scoring='accuracy',
        verbose=1,
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)
    
    # >> Mostrar mejores parámetros y score
    print("=" * 60)
    print("RESULTADOS DEL GRID SEARCH")
    print("=" * 60)
    print(f"Mejores parámetros: {grid_search.best_params_}")
    print(f"Mejor accuracy (CV): {grid_search.best_score_:.4f}")
    print("=" * 60)
    
    mlflow.log_param("best_max_depth", grid_search.best_params_['max_depth'])
    mlflow.log_param("best_max_features", grid_search.best_params_['max_features'])
    mlflow.log_param("best_min_samples_split", grid_search.best_params_['min_samples_split'])
    mlflow.log_metric("best_cv_accuracy", grid_search.best_score_)

mlflow.end_run()

# ========================================
# -> DESAFÍO 3 - Comparación de modelos
# ========================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

modelos = {
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel='rbf', random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Logistic Regression": LogisticRegression(max_iter=200, random_state=42)
}

mlflow.sklearn.autolog()

print("=" * 60)
print("COMPARACIÓN DE MODELOS")
print("=" * 60)

for nombre, modelo in modelos.items():
    with mlflow.start_run(run_name=nombre):
        # > Entrenar y predecir
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        
        mlflow.log_metric("accuracy_test", acc)
        mlflow.log_metric("f1_test", f1)
        
        print(f"   {nombre:25s} -> Accuracy: {acc:.4f} | F1: {f1:.4f}")
    
    mlflow.end_run()

print("=" * 60)
print("=> Random Forest y SVM suelen obtener los mejores resultados en Iris")

# ========================================
# -> DESAFÍO 4 - Análisis de errores
# ========================================
# ~> Inspeccionar las muestras mal clasificadas
best_model = DecisionTreeClassifier(max_depth=5, max_features=4, random_state=42)
best_model.fit(X_train, y_train)
y_pred_final = best_model.predict(X_test)

incorrect_indices = np.where(y_pred_final != y_test)[0]

print("\n" + "=" * 60)
print("ANÁLISIS DE ERRORES")
print("=" * 60)
print(f"Total de errores: {len(incorrect_indices)} de {len(y_test)} muestras")

for idx in incorrect_indices:
    real = dataset.target_names[y_test[idx]]
    pred = dataset.target_names[y_pred_final[idx]]
    print(f"   Muestra {idx}: Real={real} -> Predicho={pred}")

print("\n>> Las confusiones suelen ocurrir entre versicolor y virginica")
print("   por la similitud en sus medidas de pétalo")
print("=" * 60)

# ========================================
# -> DESAFÍO 5 - PCA y visualización 2D
# ========================================
from sklearn.decomposition import PCA

# ~> Reducir a 2 dimensiones con PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# > Gráfico 1: Proyección PCA coloreada por especie real
for i, species in enumerate(dataset.target_names):
    mask = y == i
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], label=species, alpha=0.7, edgecolors='k')
axes[0].set_xlabel('Componente Principal 1', fontsize=11)
axes[0].set_ylabel('Componente Principal 2', fontsize=11)
axes[0].set_title('PCA - Clases Reales', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# > Gráfico 2: Proyección PCA coloreada por predicción
y_pred_all = best_model.predict(X)
for i, species in enumerate(dataset.target_names):
    mask = y_pred_all == i
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], label=species, alpha=0.7, edgecolors='k')
axes[1].set_xlabel('Componente Principal 1', fontsize=11)
axes[1].set_ylabel('Componente Principal 2', fontsize=11)
axes[1].set_title('PCA - Predicciones del Modelo', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n>> Varianza explicada por PCA: {pca.explained_variance_ratio_}")
print(f">> Varianza total explicada: {sum(pca.explained_variance_ratio_):.4f}")
print("=> PCA muestra que setosa es claramente separable, versicolor y virginica se solapan")
